In [1]:
import pandas as pd
import duckdb
import os
DATABASE = "raw_data/database.db"
import sys
print(sys.executable)

/opt/miniconda3/envs/blitz_viz/bin/python


In [ ]:
con = duckdb.connect(DATABASE)

with duckdb.connect(DATABASE) as con:
    con.sql("""
        CREATE OR REPLACE TABLE auctions AS
        SELECT *
        FROM read_csv_auto('server/raw_data/auctions_cleaned/*');
    """)

    con.sql("""
        ALTER TABLE auctions
        RENAME COLUMN "order" TO pick_num;
    """)

    con.sql("""
        CREATE OR REPLACE TABLE races AS
        SELECT *
        FROM read_csv_auto('server/raw_data/race_results.csv');
    """)

    con.sql("""
        CREATE OR REPLACE TABLE draft_pool AS
        SELECT *
        FROM read_csv_auto('server/raw_data/Pok.csv')
        WHERE stage='base' AND NOT name='Egg';
    """)


In [24]:
with duckdb.connect(DATABASE) as con:
    con.sql("""
        SELECT *
        FROM auctions
        WHERE pick_num=1
        LIMIT 10
        ;
    """).show()
    
    con.sql("""
        SELECT pokemon, count(pokemon), avg(cost)
        FROM auctions
        GROUP BY pokemon
        HAVING count(pokemon) >=4
        ORDER BY count(pokemon) DESC
        ;
    """).show()

    con.sql("""
        SELECT run_id, run_number, racer_name, result, result_order
        FROM races
        WHERE placement=1
        ;
    """).show()

┌──────────┬────────────┬────────────┬───────┬───────────────┬─────────┐
│ pick_num │  pokemon   │ drafted_by │ cost  │    run_id     │ was_egg │
│  int64   │  varchar   │  varchar   │ int64 │    varchar    │  int64  │
├──────────┼────────────┼────────────┼───────┼───────────────┼─────────┤
│        1 │ Murkrow    │ Paddy      │  1900 │ 2026_01_05_r1 │    NULL │
│        1 │ Goomy      │ Primo      │  2200 │ 2026_01_08_r1 │    NULL │
│        1 │ Pancham    │ Pavan      │  1900 │ 2026_01_11_r1 │    NULL │
│        1 │ Inkay      │ Austin     │  1700 │ 2026_01_19_r1 │    NULL │
│        1 │ Horsea     │ Paddy      │  2600 │ 2026_01_23_r1 │    NULL │
│        1 │ Helioptile │ Austin     │  1700 │ 2026_01_23_r2 │    NULL │
│        1 │ Treecko    │ Jackson    │  2500 │ 2026_01_24_r1 │    NULL │
│        1 │ Onix       │ Paddy      │  2000 │ 2026_01_25_r1 │    NULL │
└──────────┴────────────┴────────────┴───────┴───────────────┴─────────┘

┌───────────┬────────────────┬────────────────────

In [25]:
most_shown = ["Oddish", "Machop", "Teddiursa", "Scyther", "Espurr"]

with duckdb.connect(DATABASE) as con:
    for mon in most_shown:
        con.sql(f"""
        SELECT pokemon, cost, drafted_by, pick_num, placement, num_picks, run_number, auctions.run_id
        FROM auctions
        JOIN races ON auctions.run_id = races.run_id AND auctions.drafted_by = races.racer_name
        WHERE pokemon='{mon}'
        ORDER BY auctions.run_id ASC;
        """).show()

┌─────────┬───────┬────────────┬──────────┬───────────┬───────────┬────────────┬───────────────┐
│ pokemon │ cost  │ drafted_by │ pick_num │ placement │ num_picks │ run_number │    run_id     │
│ varchar │ int64 │  varchar   │  int64   │   int64   │   int64   │   int64    │    varchar    │
├─────────┼───────┼────────────┼──────────┼───────────┼───────────┼────────────┼───────────────┤
│ Oddish  │  1900 │ Austin     │       40 │         4 │        64 │          1 │ 2026_01_05_r1 │
│ Oddish  │  2400 │ Primo      │       43 │         1 │        48 │          3 │ 2026_01_11_r1 │
│ Oddish  │  2200 │ Austin     │       16 │         2 │        32 │          4 │ 2026_01_19_r1 │
│ Oddish  │  3800 │ Austin     │       39 │         4 │        40 │          6 │ 2026_01_23_r2 │
│ Oddish  │  2300 │ Ely        │        5 │         6 │        48 │          7 │ 2026_01_24_r1 │
│ Oddish  │  2800 │ Ely        │       24 │         4 │        64 │          8 │ 2026_01_25_r1 │
└─────────┴───────┴───────────

In [26]:
with duckdb.connect(DATABASE) as con:
    con.sql("""
        SELECT pokemon, cost, drafted_by, pick_num, num_picks, run_number, result, result_order, placement, num_racers,  auctions.run_id, FLOOR((pick_num - 1) / (num_picks / 2)) AS buckets_2,  FLOOR((pick_num - 1) / (num_picks / 4)) AS buckets_4
        FROM auctions
        JOIN races on auctions.run_id = races.run_id and auctions.drafted_by=races.racer_name
        WHERE pokemon='Popplio'
        ORDER BY auctions.run_id ASC
        ;
    """).show()

┌─────────┬───────┬────────────┬──────────┬───────────┬────────────┬─────────┬──────────────┬───────────┬────────────┬───────────────┬───────────┬───────────┐
│ pokemon │ cost  │ drafted_by │ pick_num │ num_picks │ run_number │ result  │ result_order │ placement │ num_racers │    run_id     │ buckets_2 │ buckets_4 │
│ varchar │ int64 │  varchar   │  int64   │   int64   │   int64    │ varchar │    int64     │   int64   │   int64    │    varchar    │  double   │  double   │
├─────────┼───────┼────────────┼──────────┼───────────┼────────────┼─────────┼──────────────┼───────────┼────────────┼───────────────┼───────────┼───────────┤
│ Popplio │  4500 │ Joey       │       15 │        72 │          2 │ Sidney  │            9 │         2 │          9 │ 2026_01_08_r1 │       0.0 │       0.0 │
│ Popplio │  3500 │ Austin     │        3 │        48 │          7 │ Viola   │            5 │         5 │          6 │ 2026_01_24_r1 │       0.0 │       0.0 │
└─────────┴───────┴────────────┴──────────┴───

In [29]:
with duckdb.connect(DATABASE) as con:
    con.sql("""
        SELECT pokemon, FLOOR(AVG(cost)) AS avg_cost, count(pokemon) AS times_appeared
        FROM auctions
        GROUP BY pokemon;
        ;
    """).show()

┌────────────┬────────────────────┬────────────────┐
│  pokemon   │ floor(avg("cost")) │ count(pokemon) │
│  varchar   │       double       │     int64      │
├────────────┼────────────────────┼────────────────┤
│ Starly     │             2266.0 │              3 │
│ Noibat     │             2150.0 │              2 │
│ Mudbray    │             3300.0 │              1 │
│ Timburr    │             2150.0 │              2 │
│ Litten     │             2766.0 │              3 │
│ Snubbull   │             1875.0 │              4 │
│ Clobbopus  │             1866.0 │              3 │
│ Sandygast  │             2100.0 │              2 │
│ Joltik     │             2800.0 │              2 │
│ Litleo     │             1900.0 │              1 │
│   ·        │                ·   │              · │
│   ·        │                ·   │              · │
│   ·        │                ·   │              · │
│ Bulbasaur  │             3266.0 │              3 │
│ Turtonator │             2000.0 │           

In [45]:
with duckdb.connect(DATABASE) as con:
    # con.sql("""
    #     SELECT pokemon, FLOOR(AVG(cost)) AS avg_cost, count(pokemon) AS times_appeared
    #     FROM auctions
    #     RIGHT JOIN draft_pool ON auctions.pokemon = draft_pool.name
    #     GROUP BY pokemon;
    #     ;
    # """).show()

    # con.sql("""
    #     SELECT name
    #     FROM draft_pool
    #     ;
    # """).show(max_rows=206)

    con.sql("""
        SELECT name, COALESCE(FLOOR(AVG(cost)), 0) AS avg_cost, count(name) AS times_appeared
        FROM draft_pool
        LEFT JOIN auctions ON draft_pool.name = auctions.pokemon
        GROUP BY name
        ORDER BY times_appeared ASC;
        ;
    """).show(max_rows=203)

┌─────────────┬──────────┬────────────────┐
│    name     │ avg_cost │ times_appeared │
│   varchar   │  double  │     int64      │
├─────────────┼──────────┼────────────────┤
│ Paras       │      0.0 │              1 │
│ Nincada     │      0.0 │              1 │
│ Frillish    │      0.0 │              1 │
│ Charmander  │      0.0 │              1 │
│ Wynaut      │      0.0 │              1 │
│ Honedge     │      0.0 │              1 │
│ Igglybuff   │      0.0 │              1 │
│ Spritzee    │   1900.0 │              1 │
│ Electrike   │   2600.0 │              1 │
│ Smeargle    │   1100.0 │              1 │
│ Flabebe     │   1900.0 │              1 │
│ Beldum      │   2400.0 │              1 │
│ Varoom      │      0.0 │              1 │
│ Smoochum    │      0.0 │              1 │
│ Poochyena   │      0.0 │              1 │
│ Gothita     │      0.0 │              1 │
│ Archen      │   1900.0 │              1 │
│ Miltank     │      0.0 │              1 │
│ Feebas      │   2500.0 │      

In [30]:
# TOTAL_RUNS_QUERY = """
#     SELECT count(DISTINCT(run_id))
#     FROM races;
# """

with duckdb.connect(DATABASE) as con:
    con.sql("""
        SELECT count(DISTINCT(run_id))
        FROM races;
    """).show()

┌────────────────────────┐
│ count(DISTINCT run_id) │
│         int64          │
├────────────────────────┤
│                      8 │
└────────────────────────┘



In [6]:
with duckdb.connect(DATABASE) as con:
    con.sql("""
        SELECT ROUND(AVG(result_order),1) AS average_place, racer_name, count(run_id) AS number_of_races
        FROM races
        GROUP BY racer_name
        ORDER BY AVG(result_order) DESC
        ;
    """).show()

┌───────────────┬────────────┬─────────────────┐
│ average_place │ racer_name │ number_of_races │
│    double     │  varchar   │      int64      │
├───────────────┼────────────┼─────────────────┤
│           9.1 │ Ely        │              20 │
│           8.9 │ Jackson    │              28 │
│           8.8 │ Pavan      │              24 │
│           7.1 │ Eli        │              15 │
│           6.0 │ Joey       │               6 │
│           5.7 │ Primo      │              16 │
│           5.6 │ Paddy      │              19 │
│           5.0 │ Ian        │               1 │
│           5.0 │ Christian  │              15 │
│           4.6 │ Austin     │              19 │
├───────────────┴────────────┴─────────────────┤
│ 10 rows                            3 columns │
└──────────────────────────────────────────────┘



In [3]:
with duckdb.connect(DATABASE) as con:
    con.sql("""
        SELECT e4_order, result_order, racer_name
        FROM races
        WHERE result_order >8
        ORDER BY racer_name, result_order DESC
        ;
    """).show()

┌─────────────────────────────────────────────┬──────────────┬────────────┐
│                  e4_order                   │ result_order │ racer_name │
│                   varchar                   │    int64     │  varchar   │
├─────────────────────────────────────────────┼──────────────┼────────────┤
│ Sidney,Lucy,Glacia,Brandon,Wally,Win        │           14 │ Eli        │
│ Sidney,Phoebe,Tucker,Brandon,Steven,Loss    │           13 │ Eli        │
│ Sidney,Phoebe,Glacia,Drake,Loss             │           12 │ Eli        │
│ Spenser,Lucy,Tucker,Drake,Steven,Win        │           14 │ Ely        │
│ Sidney,Lucy,Glacia,Brandon,Steven,Win       │           14 │ Ely        │
│ Sidney,Lucy,Glacia,Brandon,Steven,Win       │           14 │ Ely        │
│ unknown,unknown,unknown,unknown,Steven,Loss │           13 │ Ely        │
│ Spenser,Lucy,Tucker,Brandon,Wally,Loss      │           13 │ Ely        │
│ Sidney,Phoebe,Tucker,Loss                   │           12 │ Ely        │
│ Spenser,Lu

In [28]:
with duckdb.connect(DATABASE) as con:
    con.sql("""
        SELECT
            result,
            COUNT(*) FILTER (WHERE result_order = 1) as "1",
            COUNT(*) FILTER (WHERE result_order = 2) as "2",
            COUNT(*) FILTER (WHERE result_order = 3) as "3",
            COUNT(*) FILTER (WHERE result_order = 4) as "4",
            COUNT(*) FILTER (WHERE result_order = 5) as "5",
            COUNT(*) FILTER (WHERE result_order = 6) as "6",
            COUNT(*) FILTER (WHERE result_order = 7) as "7",
            COUNT(*) FILTER (WHERE result_order = 8) as "8",
            COUNT(*) FILTER (WHERE result_order = 9) as "9",
            COUNT(*) FILTER (WHERE result_order = 10) as "10",
            COUNT(*) FILTER (WHERE result_order = 11) as "11",
            COUNT(*) FILTER (WHERE result_order = 12) as "12",
            COUNT(*) FILTER (WHERE result_order = 13) as "13",
            COUNT(*) FILTER (WHERE result_order = 14) as "14"
        FROM races
        GROUP BY result
        ORDER BY result;
    """).show()

┌────────────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│     result     │   1   │   2   │   3   │   4   │   5   │   6   │   7   │   8   │   9   │  10   │  11   │  12   │  13   │  14   │
│    varchar     │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │ int64 │
├────────────────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┼───────┤
│ Archie         │     0 │     0 │     0 │     0 │     0 │     0 │     0 │     3 │     0 │     0 │     0 │     0 │     0 │     0 │
│ Brandon        │     0 │     0 │     0 │     0 │     0 │     0 │     0 │     0 │     0 │     0 │     0 │     1 │     0 │     0 │
│ Brawly         │     0 │     1 │     1 │     1 │     2 │     1 │     3 │     1 │     0 │     0 │     0 │     0 │     0 │     0 │
│ Drake          │     0 │     0 │     0 │     0 │     0 │     0 │     0 │     0 │ 

In [31]:


with duckdb.connect(DATABASE) as con:
    con.sql("""
        SELECT
            result_order,
            COUNT(*) FILTER (WHERE result = 'Roxanne') as "Roxanne",
            COUNT(*) FILTER (WHERE result = 'Viola') as "Viola",
            COUNT(*) FILTER (WHERE result = 'Brawly') as "Brawly",
            COUNT(*) FILTER (WHERE result = 'Wattson') as "Wattson",
            COUNT(*) FILTER (WHERE result = 'Flannery') as "Flannery",
            COUNT(*) FILTER (WHERE result = 'Norman') as "Norman",
            COUNT(*) FILTER (WHERE result = 'Winona') as "Winona",
            COUNT(*) FILTER (WHERE result = 'Tate & Liza') as "Tate & Liza",
            COUNT(*) FILTER (WHERE result = 'Juan & Wallace') as "Juan & Wallace",
            COUNT(*) FILTER (WHERE result = 'Archie') as "Archie",
            COUNT(*) FILTER (WHERE result = 'Maxie') as "Maxie",
            COUNT(*) FILTER (WHERE result = 'Sidney') as "Sidney",
            COUNT(*) FILTER (WHERE result = 'Spenser') as "Spenser",
            COUNT(*) FILTER (WHERE result = 'Lucy') as "Lucy",
            COUNT(*) FILTER (WHERE result = 'Phoebe') as "Phoebe",
            COUNT(*) FILTER (WHERE result = 'Glacia') as "Glacia",
            COUNT(*) FILTER (WHERE result = 'Tucker') as "Tucker",
            COUNT(*) FILTER (WHERE result = 'Drake') as "Drake",
            COUNT(*) FILTER (WHERE result = 'Brandon') as "Brandon",
            COUNT(*) FILTER (WHERE result = 'Steven') as "Steven",
            COUNT(*) FILTER (WHERE result = 'Wally') as "Wally",
        FROM races
        GROUP BY result_order
        ORDER BY result_order;
    """).show()

┌──────────────┬─────────┬───────┬────────┬─────────┬──────────┬────────┬────────┬─────────────┬────────────────┬────────┬───────┬────────┬─────────┬───────┬────────┬────────┬────────┬───────┬─────────┬────────┬───────┐
│ result_order │ Roxanne │ Viola │ Brawly │ Wattson │ Flannery │ Norman │ Winona │ Tate & Liza │ Juan & Wallace │ Archie │ Maxie │ Sidney │ Spenser │ Lucy  │ Phoebe │ Glacia │ Tucker │ Drake │ Brandon │ Steven │ Wally │
│    int64     │  int64  │ int64 │ int64  │  int64  │  int64   │ int64  │ int64  │    int64    │     int64      │ int64  │ int64 │ int64  │  int64  │ int64 │ int64  │ int64  │ int64  │ int64 │  int64  │ int64  │ int64 │
├──────────────┼─────────┼───────┼────────┼─────────┼──────────┼────────┼────────┼─────────────┼────────────────┼────────┼───────┼────────┼─────────┼───────┼────────┼────────┼────────┼───────┼─────────┼────────┼───────┤
│            2 │       0 │     0 │      1 │       1 │        0 │      3 │      0 │           1 │              2 │      0

In [2]:

with duckdb.connect(DATABASE) as con:
    con.sql("""
    SELECT run_number, racer_name, result, result_order, game_version
    FROM races
    WHERE racer_name='Paddy'
    ORDER BY result_order DESC;
    """).show()

┌────────────┬────────────┬────────────────┬──────────────┬──────────────┐
│ run_number │ racer_name │     result     │ result_order │ game_version │
│   int64    │  varchar   │    varchar     │    int64     │    double    │
├────────────┼────────────┼────────────────┼──────────────┼──────────────┤
│         29 │ Paddy      │ Steven         │           14 │         8.31 │
│         32 │ Paddy      │ Steven         │           14 │         8.31 │
│         21 │ Paddy      │ Spenser        │            9 │          8.2 │
│         15 │ Paddy      │ Archie         │            8 │         7.91 │
│         25 │ Paddy      │ Wattson        │            7 │          8.3 │
│          6 │ Paddy      │ Tate & Liza    │            6 │          7.8 │
│         14 │ Paddy      │ Winona         │            6 │         7.91 │
│          4 │ Paddy      │ Flannery       │            5 │          7.7 │
│          1 │ Paddy      │ Juan & Wallace │            5 │          7.4 │
│          8 │ Paddy     

In [22]:
with duckdb.connect(DATABASE) as con:
    con.sql("""            
        SELECT *
        FROM draft_pool
        WHERE stage='base' AND NOT name='Egg' AND is_baby IS NULL;
    """).show(max_rows=200)


┌────────────┬──────────────────┬──────────────────┬─────────┬──────────────────┬───────────────┬────────────────┬───────┬────────┬─────────┬───────────┬────────────┬───────┬──────────────────┬─────────┬─────────┐
│ dex_number │       name       │       type       │  stage  │     ability1     │   ability2    │ hidden_ability │  hp   │ attack │ defense │ sp_attack │ sp_defense │ speed │ evolution_method │  mega   │ is_baby │
│   int64    │     varchar      │     varchar      │ varchar │     varchar      │    varchar    │    varchar     │ int64 │ int64  │  int64  │   int64   │   int64    │ int64 │     varchar      │ varchar │ boolean │
├────────────┼──────────────────┼──────────────────┼─────────┼──────────────────┼───────────────┼────────────────┼───────┼────────┼─────────┼───────────┼────────────┼───────┼──────────────────┼─────────┼─────────┤
│        359 │ Absol            │ Dark             │ base    │ Pressure         │ Super Luck    │ Justified      │    65 │    130 │      60 │   

In [7]:
with duckdb.connect(DATABASE) as con:
    con.sql("""            
        SELECT DISTINCT pokemon
        FROM draft_pool
        RIGHT JOIN auctions ON draft_pool.name = auctions.pokemon
        WHERE draft_pool.name IS NULL
    """).show()

┌─────────────┐
│   pokemon   │
│   varchar   │
├─────────────┤
│ Turtonator  │
│ Hawlucha    │
│ Absol       │
│ Falinks     │
│ Miltank     │
│ Klawf       │
│ Stonjourner │
│ Happiny     │
│ Bombirdier  │
│ Wynaut      │
├─────────────┤
│   10 rows   │
└─────────────┘



In [38]:
with duckdb.connect(DATABASE) as con:
    con.sql("""            
        SELECT racer_name, eevee_picked, run_number
        FROM races
        WHERE eevee_picked IS NOT NULL AND racer_name='Jackson'
    """).show()

┌────────────┬──────────────┬────────────┐
│ racer_name │ eevee_picked │ run_number │
│  varchar   │   varchar    │   int64    │
├────────────┼──────────────┼────────────┤
│ Jackson    │ Sylveon      │          7 │
│ Jackson    │ Vaporeon     │          6 │
│ Jackson    │ Sylveon      │          5 │
│ Jackson    │ Umbreon      │          4 │
│ Jackson    │ Vaporeon     │          2 │
│ Jackson    │ Sylveon      │         13 │
│ Jackson    │ Sylveon      │         14 │
│ Jackson    │ Vaporeon     │         15 │
│ Jackson    │ Sylveon      │         16 │
│ Jackson    │ Umbreon      │         17 │
│ Jackson    │ Jolteon      │         18 │
│ Jackson    │ Leafeon      │         21 │
│ Jackson    │ Sylveon      │         22 │
│ Jackson    │ Vaporeon     │         23 │
│ Jackson    │ Jolteon      │         24 │
│ Jackson    │ Umbreon      │         25 │
│ Jackson    │ Vaporeon     │         26 │
│ Jackson    │ Jolteon      │         31 │
├────────────┴──────────────┴────────────┤
│ 18 rows  

In [17]:
with duckdb.connect(DATABASE) as con:
    con.sql("""            
        SELECT mvp, racer_name, mvp_kos, races.run_id, base_evo, COALESCE(cost,null) AS cost
        FROM races
        LEFT JOIN pok ON races.mvp = pok.name
        LEFT JOIN auctions ON pok.base_evo = auctions.pokemon AND races.run_id = auctions.run_id
        WHERE mvp IS NOT NULL
        ORDER BY mvp_kos DESC
        LIMIT 10;
    """).show()

┌───────────┬────────────┬─────────┬───────────────┬──────────┬───────┐
│    mvp    │ racer_name │ mvp_kos │    run_id     │ base_evo │ cost  │
│  varchar  │  varchar   │  int64  │    varchar    │ varchar  │ int64 │
├───────────┼────────────┼─────────┼───────────────┼──────────┼───────┤
│ Blaziken  │ Ely        │      15 │ 2026_03_13_r1 │ Torchic  │  3400 │
│ Mamoswine │ Paddy      │      15 │ 2026_03_14_r1 │ Swinub   │  3100 │
│ Mamoswine │ Paddy      │      15 │ 2026_03_10_r1 │ Swinub   │  2200 │
│ Appletun  │ Ely        │      15 │ 2026_03_01_r1 │ Applin   │  2800 │
│ Vaporeon  │ Jackson    │      14 │ 2026_03_15_r1 │ Eevee    │  NULL │
│ Mantine   │ Ely        │      14 │ 2026_03_14_r1 │ Mantyke  │  2300 │
│ Ludicolo  │ Pavan      │      13 │ 2026_03_08_r1 │ Lotad    │  2100 │
│ Magmortar │ Ely        │      12 │ 2026_03_15_r1 │ Magby    │  2700 │
│ Serperior │ Eli        │      10 │ 2026_03_10_r1 │ Snivy    │  2800 │
└───────────┴────────────┴─────────┴───────────────┴──────────┴─

In [17]:
with duckdb.connect(DATABASE) as con:
    con.sql("""
        SELECT mons, count
        FROM (
            SELECT 
                mons,COUNT(*) AS count, RANK() OVER (ORDER BY COUNT(*) DESC) AS rnk
            FROM (
                SELECT unnest(string_split(hof_team, ',')) AS mons
                FROM races
            )
            GROUP BY mons
        )
        WHERE rnk <= 10
        ORDER BY rnk;
    """).show()

┌────────────┬───────┐
│    mons    │ count │
│  varchar   │ int64 │
├────────────┼───────┤
│ Sylveon    │     3 │
│ Metagross  │     2 │
│ Golem      │     2 │
│ Florges    │     2 │
│ Espeon     │     2 │
│ Delphox    │     2 │
│ Toxicroak  │     2 │
│ Mamoswine  │     2 │
│ Vaporeon   │     2 │
│ Staraptor  │     2 │
│ Salamence  │     2 │
│ Coalossal  │     2 │
│ Jolteon    │     2 │
│ Ribombee   │     2 │
│ Blaziken   │     2 │
│ Aromatisse │     2 │
├────────────┴───────┤
│ 16 rows  2 columns │
└────────────────────┘



In [2]:
with duckdb.connect(DATABASE) as con:
    con.sql("""
        SELECT *
        FROM pok
    """).show()

┌────────────┬───────────┬──────────────┬─────────┬──────────────────┬──────────────┬────────────────┬───────┬────────┬─────────┬───────────┬────────────┬───────┬───────────────────────────┬─────────┬─────────┬──────────┐
│ dex_number │   name    │     type     │  stage  │     ability1     │   ability2   │ hidden_ability │  hp   │ attack │ defense │ sp_attack │ sp_defense │ speed │     evolution_method      │  mega   │ is_baby │ base_evo │
│   int64    │  varchar  │   varchar    │ varchar │     varchar      │   varchar    │    varchar     │ int64 │ int64  │  int64  │   int64   │   int64    │ int64 │          varchar          │ varchar │ boolean │ varchar  │
├────────────┼───────────┼──────────────┼─────────┼──────────────────┼──────────────┼────────────────┼───────┼────────┼─────────┼───────────┼────────────┼───────┼───────────────────────────┼─────────┼─────────┼──────────┤
│        698 │ Amaura    │ Rock/Ice     │ base    │ Refrigerate      │ NULL         │ Snow Warning   │    77 │  